In [1]:
import os
!git clone https://github.com/bencejdanko/bert-tweeteval

# ensure latest
os.chdir('/content/bert-tweeteval')
!cd /content/bert-tweeteval && git pull

Cloning into 'bert-tweeteval'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 206 (delta 131), reused 115 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 794.59 KiB | 3.35 MiB/s, done.
Resolving deltas: 100% (131/131), done.
Already up to date.


In [2]:
# copy over package
PACKAGE = "src"

import sys
sys.path.append(f"/content/bert-tweeteval/{PACKAGE}")

In [3]:
import pandas as pd
import json
from transformers import AutoTokenizer, pipeline
from download import download_and_split_dataset

In [4]:
id_labels = {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"}
train_df, val_df, test_df = download_and_split_dataset()
test_df['label_str'] = test_df['label'].map(id_labels)


README.md: 0.00B [00:00, ?B/s]

emotion/train-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

emotion/test-00000-of-00001.parquet:   0%|          | 0.00/105k [00:00<?, ?B/s]

emotion/validation-00000-of-00001.parque(…):   0%|          | 0.00/28.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/374 [00:00<?, ? examples/s]

In [5]:
with open("/content/bert-tweeteval/src/error_analysis_indices.json", "r") as f:
    indices = json.load(f)
    both_wrong_zero = indices['both_wrong']
    bert_right_zero = indices['bert_right_rob_wrong']
    rob_right_zero = indices['rob_right_bert_wrong']
print("Loaded zero-shot error indices.")


Loaded zero-shot error indices.


In [6]:
model_bert_id = "bdanko/bert-tweeteval-distilbert"
model_rob_id = "bdanko/bert-tweeteval-distilroberta"

pipe_bert = pipeline("text-classification", model=model_bert_id)
pipe_rob = pipeline("text-classification", model=model_rob_id)

tokenizer_bert = AutoTokenizer.from_pretrained(model_bert_id)
tokenizer_rob = AutoTokenizer.from_pretrained(model_rob_id)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
def predict_batch(texts, pipe):
    preds = pipe(texts, truncation=True, max_length=512)
    return [p['label'].lower() for p in preds]

print("Generating predictions...")
texts = test_df['text'].tolist()
test_df['distilbert_pred'] = predict_batch(texts, pipe_bert)
test_df['distilroberta_pred'] = predict_batch(texts, pipe_rob)

Generating predictions...


In [8]:
def print_sample_analysis(indices, title):
    if not indices:
        return
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")

    subset = test_df.loc[indices]
    for idx, row in subset.iterrows():
        text = row['text']
        true_label = row['label_str']
        pred_bert = row['distilbert_pred']
        pred_rob = row['distilroberta_pred']

        tokens_bert = tokenizer_bert.tokenize(text)
        tokens_rob = tokenizer_rob.tokenize(text)

        print(f"Text: {text}")
        print(f"True: {true_label} | Fine-Tuned DistilBERT Pred: {pred_bert} | Fine-Tuned DistilRoBERTa Pred: {pred_rob}")
        print(f"WordPiece (DistilBERT): {tokens_bert}")
        print(f"BPE (DistilRoBERTa): {tokens_rob}")
        print("-" * 50)

print_sample_analysis(both_wrong_zero, "Samples BOTH zero-shot models misclassified")
print_sample_analysis(bert_right_zero, "Samples zero-shot DistilBERT got right, but DistilRoBERTa missed")
print_sample_analysis(rob_right_zero, "Samples zero-shot DistilRoBERTa got right, but DistilBERT missed")



Samples BOTH zero-shot models misclassified
Text: @user Interesting choice of words... Are you confirming that governments fund #terrorism? Bit of an open door, but still...
True: anger | Fine-Tuned DistilBERT Pred: label_0 | Fine-Tuned DistilRoBERTa Pred: label_0
WordPiece (DistilBERT): ['@', 'user', 'interesting', 'choice', 'of', 'words', '.', '.', '.', 'are', 'you', 'confirming', 'that', 'governments', 'fund', '#', 'terrorism', '?', 'bit', 'of', 'an', 'open', 'door', ',', 'but', 'still', '.', '.', '.']
BPE (DistilRoBERTa): ['@', 'user', 'ĠInteresting', 'Ġchoice', 'Ġof', 'Ġwords', '...', 'ĠAre', 'Ġyou', 'Ġconfirming', 'Ġthat', 'Ġgovernments', 'Ġfund', 'Ġ#', 'terrorism', '?', 'ĠBit', 'Ġof', 'Ġan', 'Ġopen', 'Ġdoor', ',', 'Ġbut', 'Ġstill', '...']
--------------------------------------------------
Text: @user Welcome to #MPSVT! We are delighted to have you! #grateful #MPSVT #relationships
True: joy | Fine-Tuned DistilBERT Pred: label_1 | Fine-Tuned DistilRoBERTa Pred: label_1
WordPiece 

In [9]:
both_wrong_ft = test_df[(test_df['distilbert_pred'] != test_df['label_str']) & (test_df['distilroberta_pred'] != test_df['label_str'])].head(8)
bert_right_rob_wrong_ft = test_df[(test_df['distilbert_pred'] == test_df['label_str']) & (test_df['distilroberta_pred'] != test_df['label_str'])].head(8)
rob_right_bert_wrong_ft = test_df[(test_df['distilroberta_pred'] == test_df['label_str']) & (test_df['distilbert_pred'] != test_df['label_str'])].head(8)

print_sample_analysis(both_wrong_ft.index.tolist(), "Misclassified by BOTH Fine-Tuned Models")
print_sample_analysis(bert_right_rob_wrong_ft.index.tolist(), "Fine-Tuned DistilBERT Right, DistilRoBERTa Wrong")
print_sample_analysis(rob_right_bert_wrong_ft.index.tolist(), "Fine-Tuned DistilRoBERTa Right, DistilBERT Wrong")



Misclassified by BOTH Fine-Tuned Models
Text: #Deppression is real. Partners w/ #depressed people truly dont understand the depth in which they affect us. Add in #anxiety &amp;makes it worse
True: sadness | Fine-Tuned DistilBERT Pred: label_3 | Fine-Tuned DistilRoBERTa Pred: label_3
WordPiece (DistilBERT): ['#', 'de', '##pp', '##ress', '##ion', 'is', 'real', '.', 'partners', 'w', '/', '#', 'depressed', 'people', 'truly', 'don', '##t', 'understand', 'the', 'depth', 'in', 'which', 'they', 'affect', 'us', '.', 'add', 'in', '#', 'anxiety', '&', 'amp', ';', 'makes', 'it', 'worse']
BPE (DistilRoBERTa): ['#', 'De', 'pp', 'ression', 'Ġis', 'Ġreal', '.', 'ĠPartners', 'Ġw', '/', 'Ġ#', 'dep', 'ressed', 'Ġpeople', 'Ġtruly', 'Ġdont', 'Ġunderstand', 'Ġthe', 'Ġdepth', 'Ġin', 'Ġwhich', 'Ġthey', 'Ġaffect', 'Ġus', '.', 'ĠAdd', 'Ġin', 'Ġ#', 'an', 'xiety', 'Ġ&', 'amp', ';', 'makes', 'Ġit', 'Ġworse']
--------------------------------------------------
Text: @user Interesting choice of words... Are you conf